# ComfyUI on Colab, with models in Google DriveComfyUI runs on Colab's GPU; your checkpoints, outputs and workflows live inGoogle Drive so they survive the runtime being recycled.**Before you start:** `Runtime > Change runtime type > Hardware accelerator > GPU`.Run the cells in order. Cell 6 is the one that keeps running — it serves the UIand prints a link you can open. Stopping that cell stops ComfyUI.

In [ ]:
#@title 1. Check the GPUimport subprocessr = subprocess.run(    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],    capture_output=True, text=True,)if r.returncode != 0:    print("No GPU attached to this runtime.")    print("Fix: Runtime > Change runtime type > Hardware accelerator > GPU, then re-run.")    print()    print("ComfyUI still works on CPU, but SD1.5 takes minutes per image")    print("and SDXL is impractical. Getting a GPU is worth the detour.")else:    print("GPU:", r.stdout.strip())

In [ ]:
#@title 2. Mount Drive and choose where things livefrom google.colab import driveimport osdrive.mount("/content/drive")DRIVE_ROOT = "/content/drive/MyDrive/ComfyUI"  #@param {type:"string"}COMFY = "/content/ComfyUI"# Created once, then reused by every future session.for sub in ("models/checkpoints", "models/clip", "models/clip_vision",            "models/controlnet", "models/diffusion_models", "models/embeddings",            "models/loras", "models/text_encoders", "models/unet",            "models/upscale_models", "models/vae", "output", "input", "workflows"):    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)print("Persistent folder:", DRIVE_ROOT)

In [ ]:
#@title 3. Install ComfyUI# ComfyUI's own code stays on the runtime's local disk rather than Drive: it is# thousands of small files, and Drive's FUSE mount makes both the install and# every startup dramatically slower. Only the things worth keeping go to Drive.import osif os.path.isdir(f"{COMFY}/.git"):    !cd {COMFY} && git pull --ff-onlyelse:    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY}# Colab already ships a CUDA-enabled torch. Install ComfyUI's other requirements# on top of it and leave torch alone -- reinstalling it here is the usual way# people end up with a broken CUDA setup on Colab.!cd {COMFY} && pip install -q -r requirements.txtprint("ComfyUI installed at", COMFY)

In [ ]:
#@title 4. Point ComfyUI's folders at Driveimport os, shutilLINKS = {    f"{COMFY}/models/checkpoints":          f"{DRIVE_ROOT}/models/checkpoints",    f"{COMFY}/models/clip":                 f"{DRIVE_ROOT}/models/clip",    f"{COMFY}/models/clip_vision":          f"{DRIVE_ROOT}/models/clip_vision",    f"{COMFY}/models/controlnet":           f"{DRIVE_ROOT}/models/controlnet",    f"{COMFY}/models/diffusion_models":     f"{DRIVE_ROOT}/models/diffusion_models",    f"{COMFY}/models/embeddings":           f"{DRIVE_ROOT}/models/embeddings",    f"{COMFY}/models/loras":                f"{DRIVE_ROOT}/models/loras",    f"{COMFY}/models/text_encoders":        f"{DRIVE_ROOT}/models/text_encoders",    f"{COMFY}/models/unet":                 f"{DRIVE_ROOT}/models/unet",    f"{COMFY}/models/upscale_models":       f"{DRIVE_ROOT}/models/upscale_models",    f"{COMFY}/models/vae":                  f"{DRIVE_ROOT}/models/vae",    f"{COMFY}/output":                      f"{DRIVE_ROOT}/output",    f"{COMFY}/input":                       f"{DRIVE_ROOT}/input",}for local, target in LINKS.items():    os.makedirs(target, exist_ok=True)    # A fresh clone leaves real (near-empty) directories here; replace them with    # symlinks. Anything already linked is repointed rather than duplicated.    if os.path.islink(local):        os.unlink(local)    elif os.path.isdir(local):        shutil.rmtree(local)    os.makedirs(os.path.dirname(local), exist_ok=True)    os.symlink(target, local)    print(f"{local.replace(COMFY, '.')}  ->  Drive")

In [ ]:
#@title 5. Get a checkpoint (skipped if Drive already has it)MODEL = "sd15"  #@param ["sd15", "sdxl", "sdxl-turbo", "none"]CATALOG = {    "sd15": ("v1-5-pruned-emaonly-fp16.safetensors",             "https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/"             "resolve/main/v1-5-pruned-emaonly-fp16.safetensors"),    "sdxl": ("sd_xl_base_1.0.safetensors",             "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/"             "resolve/main/sd_xl_base_1.0.safetensors"),    "sdxl-turbo": ("sd_xl_turbo_1.0_fp16.safetensors",             "https://huggingface.co/stabilityai/sdxl-turbo/"             "resolve/main/sd_xl_turbo_1.0_fp16.safetensors"),}import osif MODEL == "none":    print("Skipping download.")else:    fname, url = CATALOG[MODEL]    dest = f"{DRIVE_ROOT}/models/checkpoints/{fname}"    if os.path.exists(dest):        print(f"{fname} is already in Drive, nothing to download.")    else:        # Download beside the target, then rename, so an interrupted transfer is        # never left looking like a complete checkpoint.        print(f"Downloading {fname} into Drive (this is a few GB, once ever)...")        !wget -q --show-progress -c -O "{dest}.part" "{url}" && mv "{dest}.part" "{dest}"        print("Saved to", dest)print()print("Checkpoints visible to ComfyUI:")for f in sorted(os.listdir(f"{DRIVE_ROOT}/models/checkpoints")):    print("  ", f)

In [ ]:
#@title 6. Launch ComfyUI  { display-mode: "form" }# This cell keeps running for as long as the server is up. Watch for the# trycloudflare.com link it prints, then open that in a new tab.PORT = 8188!wget -qc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb!dpkg -i cloudflared-linux-amd64.deb 2>/dev/null || apt-get -qq -f install -yimport subprocess, threading, retunnel = subprocess.Popen(    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,)def announce():    for line in tunnel.stdout:        found = re.search(r"https://[-\w]+\.trycloudflare\.com", line)        if found:            print("\n" + "=" * 64)            print("  Open ComfyUI here:", found.group(0))            print("  (give it a few seconds -- wait for 'To see the GUI go to' below)")            print("=" * 64 + "\n")threading.Thread(target=announce, daemon=True).start()!cd {COMFY} && python main.py --port {PORT}

## Notes**Next time round.** Drive keeps the checkpoints, so a later session is justcells 1-4 and 6 — cell 5 finds the model already there and skips the download.**Where your images go.** `output/` is symlinked to Drive, so anything yougenerate is in `MyDrive/ComfyUI/output` without exporting.**Saving workflows.** Save the workflow JSON into `MyDrive/ComfyUI/workflows`so it persists too; ComfyUI's own menu defaults to browser storage, which theruntime does not keep.**Colab disconnects.** Free Colab reclaims idle runtimes and caps GPU hours.When that happens the tunnel URL dies — re-run the cells and you get a new one.Nothing in Drive is lost.**The link is public.** A `trycloudflare.com` URL is reachable by anyone whohas it, and ComfyUI has no password. Don't share it, and stop cell 6 when done.